## Forecasting Short-Term Shell Congestion Using Regression

This notebook estimates **short-term congestion trends** in each orbital shell (KMeans cluster).  
It uses only the information inside the **current dataset snapshot**.

### How it works
- Satellites are assigned to shells using **KMeans (CLUSTER)**.
- Each satellite has an **EPOCH date**, which tells when its orbital data was last updated.
- We group satellites by **CLUSTER + EPOCH (day)** to get:
  
  **“How many satellites were updated in this shell on this date?”**

- Using these counts, we fit a simple regression line to understand the **trend**:
  - Rising → congestion increasing  
  - Stable → congestion steady  
  - Falling → congestion reducing  

### Important Note
The regression **does not predict exact future satellite counts**.  
It only shows the **trend direction** inside the current dataset, which helps us judge if an orbital shell is:

- **Safe**
- **Moderate Risk**
- **High Risk**

for a new satellite launch based on **today’s congestion and trend**.

This makes the forecasting realistic, lightweight, and reliable without needing long-term historical data.


In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

#### Import necessary datasets

In [24]:
cleaned_df = pd.read_csv('../data/03_engineered/satellites_engineered.csv')

# Any of the scaled / unscaled K means O/P DF can be used
clustered_df = pd.read_csv('../data/05_labeled/satellites_unscaled_labeled.csv')

In [25]:
cleaned_df.head(1)

,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,ORBIT_PERIOD_SEC,SEMI_MAJOR_AXIS,ORBIT_HEIGHT,PERIGEE,APOGEE,ORBITAL_SPEED,AGE_SINCE_LAUNCH,SAT_TYPE
0,2025-11-24 00:38:47.626368,13.763307,0.00263,90.2213,67.0218,189.7579,232.6051,900,4320,0.000861,0.000009,0.0,6277.560908,7354.997884,983.997884,7335.654239,7374.341528,7.361687,61.898,C


In [26]:
clustered_df.head(1)

,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,...,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z,CLUSTER
0,2025-11-24 00:38:47.626368,13.763307,0.00263,90.2213,67.0218,189.7579,232.6051,900,4320,0.000861,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3


---
### Extract required columns in each dataset

In [27]:
cleaned_df = cleaned_df[['EPOCH', 'NORAD_CAT_ID']]
clustered_df = clustered_df[['EPOCH', 'NORAD_CAT_ID', 'CLUSTER']]


In [28]:
cleaned_df.head(2)

,EPOCH,NORAD_CAT_ID
0,2025-11-24 00:38:47.626368,900
1,2025-11-23 19:18:27.313056,902


In [29]:
clustered_df.head(2)

,EPOCH,NORAD_CAT_ID,CLUSTER
0,2025-11-24 00:38:47.626368,900,3
1,2025-11-23 19:18:27.313056,902,3


---
### merge both DF on 'NORAD_CAT_ID'

In [30]:
merged_df = cleaned_df.merge(clustered_df[['NORAD_CAT_ID', 'CLUSTER']], on = 'NORAD_CAT_ID', how='left')

In [31]:
merged_df.head()

,EPOCH,NORAD_CAT_ID,CLUSTER
0,2025-11-24 00:38:47.626368,900,3
1,2025-11-23 19:18:27.313056,902,3
2,2025-11-23 16:55:04.202400,1512,3
3,2025-11-23 22:01:14.414880,1520,3
4,2025-11-23 23:24:19.305216,2826,4


In [32]:
merged_df.shape

(12603, 3)

In [34]:
merged_df['CLUSTER'].value_counts().sort_values()

CLUSTER
3    1126
4    1964
1    2273
0    2398
2    4842
Name: count, dtype: int64

---
### Prepare time column
- Convert epoch to DateTime object
- Extract the day (not time)

In [36]:
merged_df['EPOCH'] = pd.to_datetime(merged_df['EPOCH'])

In [38]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12603 entries, 0 to 12602
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   EPOCH         12603 non-null  datetime64[ns]
 1   NORAD_CAT_ID  12603 non-null  int64         
 2   CLUSTER       12603 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 295.5 KB


In [41]:
merged_df['EPOCH'] = merged_df['EPOCH'].dt.floor('D')

In [42]:
merged_df.head()

,EPOCH,NORAD_CAT_ID,CLUSTER
0,2025-11-24,900,3
1,2025-11-23,902,3
2,2025-11-23,1512,3
3,2025-11-23,1520,3
4,2025-11-23,2826,4


In [49]:
merged_df.groupby(['EPOCH', 'CLUSTER']).size().reset_index(name='Satellite Count')

,EPOCH,CLUSTER,Satellite Count
0,2025-11-08,2,1
1,2025-11-12,0,2
2,2025-11-13,0,1
3,2025-11-14,0,1
4,2025-11-16,0,1
5,2025-11-18,0,2
6,2025-11-18,2,1
7,2025-11-19,3,1
8,2025-11-20,0,7
9,2025-11-20,1,7
